# High Frequency Trading Model
Implementation of a Deep Q-Learning trading model using Interactive Brokers data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import pandas_ta as ta
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import logging
import os
import gym
from gym import spaces
from typing import Tuple, Dict
from pathlib import Path
from tqdm.auto import tqdm

# Add visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style and configuration
sns.set_style('darkgrid')
plt.style.use('default')
%matplotlib inline

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

/home/chiqo/Documents/AI/High-Frequency-Trading-Model-with-IB/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Trading Environment
Define the Forex trading environment for reinforcement learning

In [ ]:
class ForexTradingEnv(gym.Env):
    def __init__(self, df_lower: pd.DataFrame, df_higher: pd.DataFrame, initial_balance: float = 10000):
        super(ForexTradingEnv, self).__init__()
        
        self.df_lower = df_lower    # Lower timeframe data (e.g., M5, M15)
        self.df_higher = df_higher  # Higher timeframe data (e.g., H4, D1)
        self.initial_balance = initial_balance
        self.current_step = 0
        self.position = None
        self.trade_count = 0
        
        # Define action and observation spaces
        self.action_space = spaces.Discrete(4)  # Buy, Sell, Close, Hold
        
        # Observation space includes indicators from both timeframes
        self.observation_space = spaces.Box(
            low=-np.inf, 
            high=np.inf, 
            shape=(10,),  # [position, higher_sma, higher_adx, higher_close, lower_rsi, lower_macd, lower_signal, lower_close]
            dtype=np.float32
        )

    def _get_market_bias(self) -> str:
        """Determine market bias using higher timeframe data"""
        current_data = self.df_higher.iloc[self.current_step]
        
        # Get SMA and ADX values
        sma_value = current_data['SMA_20']
        adx_value = current_data['ADX_14']
        close_price = current_data['Close']
        
        if adx_value > 25:  # Strong trend
            if close_price > sma_value:
                return 'bullish'
            elif close_price < sma_value:
                return 'bearish'
        return 'neutral'

    def _check_entry_signals(self) -> dict:
        """Check entry signals on lower timeframe"""
        current_data = self.df_lower.iloc[self.current_step]
        previous_data = self.df_lower.iloc[self.current_step - 1] if self.current_step > 0 else current_data
        
        rsi = current_data['RSI_7']
        prev_rsi = previous_data['RSI_7']
        macd = current_data['MACD_8_17_6']
        macd_signal = current_data['MACDs_8_17_6']
        prev_macd = previous_data['MACD_8_17_6']
        prev_signal = previous_data['MACDs_8_17_6']
        
        signals = {
            'buy': (rsi > 30 and prev_rsi <= 30) or  # RSI crosses above 30
                   (macd > macd_signal and prev_macd <= prev_signal),  # MACD crossover
            'sell': (rsi < 70 and prev_rsi >= 70) or  # RSI crosses below 70
                    (macd < macd_signal and prev_macd >= prev_signal)  # MACD crossunder
        }
        return signals

    def _get_state(self) -> np.array:
        """Create state vector combining both timeframes"""
        higher_data = self.df_higher.iloc[self.current_step]
        lower_data = self.df_lower.iloc[self.current_step]
        
        position_flag = 0 if self.position is None else (1 if self.position == 'long' else -1)
        
        state = np.array([
            position_flag,
            higher_data['SMA_20'],
            higher_data['ADX_14'],
            higher_data['Close'],
            lower_data['RSI_7'],
            lower_data['MACD_8_17_6'],
            lower_data['MACDs_8_17_6'],
            lower_data['Close']
        ], dtype=np.float32)
        
        return state

    def step(self, action: int) -> Tuple[np.array, float, bool, Dict]:
        """Execute one trading step"""
        current_price = self.df_lower.iloc[self.current_step]['Close']
        bias = self._get_market_bias()
        signals = self._check_entry_signals()
        reward = 0
        
        # Execute trading action
        if action == 0:  # Buy
            if self.position is None and bias == 'bullish' and signals['buy']:
                self.position = 'long'
                self.trade_count += 1
        elif action == 1:  # Sell
            if self.position is None and bias == 'bearish' and signals['sell']:
                self.position = 'short'
                self.trade_count += 1
        elif action == 2:  # Close
            if self.position is not None:
                self.position = None
        # action == 3 is Hold
        
        # Move to next step
        self.current_step += 1
        done = self.current_step >= len(self.df_lower) - 1
        
        # Calculate reward based on price movement
        if not done and self.position is not None:
            next_price = self.df_lower.iloc[self.current_step]['Close']
            price_change = (next_price - current_price) / current_price
            reward = price_change if self.position == 'long' else -price_change
        
        info = {
            'trade_count': self.trade_count,
            'bias': bias,
            'position': self.position
        }
        
        return self._get_state(), reward, done, info

    def reset(self):
        self.current_step = 0
        self.position = None
        self.trade_count = 0
        return self._get_state()

: 

## Configuration
Define project paths and model parameters

In [ ]:
# Project paths
BASE_DIR = Path().absolute()
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

# Create directories if they don't exist
MODELS_DIR.mkdir(exist_ok=True)

# RL Configuration
RL_CONFIG = {
    'learning_rate': 0.001,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.01,
    'epsilon_decay': 0.995,
    'batch_size': 32
}

# Technical Indicators Configuration
TECHNICAL_INDICATORS = {
    'RSI': {'length': 7},
    'MACD': {'fast': 8, 'slow': 17, 'signal': 6},
    'BB': {'length': 10, 'std': 2},
    'ATR': {'length': 7},
    'CCI': {'length': 10},  # Commodity Channel Index
    'VHF': {'length': 14},  # Vertical Horizontal Filter
    'ERI': {'length': 7},   # Elder Ray Index
    'ADX': {'length': 10}   # Average Directional Index
}

: 

## Data Processing
Implement data loading and technical analysis

In [ ]:
class DataProcessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        print("\n[DataProcessor] Initialized")

    def load_data(self, filepath: str) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print(f"[DataProcessor] Loading data from {filepath}")
            expected_columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
            
            # Show progress while reading large files
            print("\n[DataProcessor] Reading data chunks:")
            chunks = pd.read_csv(filepath, 
                              header=None,
                              names=expected_columns,
                              sep=r'\s+',
                              chunksize=1000)
            
            df_chunks = []
            for chunk in tqdm(chunks, desc="Reading data", ncols=100):
                df_chunks.append(chunk)
            df = pd.concat(df_chunks)
            
            print(f"\n[DataProcessor] Raw data shape: {df.shape}")
            
            if df.empty:
                raise ValueError("Empty dataframe loaded")
            
            print("\n[DataProcessor] Processing steps:")
            
            # Time conversion
            print("1. Converting time...")
            df['Time'] = pd.to_datetime(df['Time'], format='mixed')
            
            # Numeric conversion
            print("2. Converting numeric columns...")
            numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
            for col in numeric_columns:
                df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
            
            # Data cleaning
            print("3. Cleaning data...")
            initial_rows = len(df)
            df = df.dropna()
            df = df.drop_duplicates(subset=['Time'], keep='first')
            
            # Index setting
            print("4. Setting index...")
            df.set_index('Time', inplace=True)
            df.sort_index(inplace=True)
            
            # Print summary
            print(f"\n[DataProcessor] Data loading completed:")
            print(f" - Initial rows: {initial_rows}")
            print(f" - Final rows: {len(df)}")
            print(f" - Columns: {', '.join(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error loading data: {str(e)}")
            raise

    def add_technical_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print("[DataProcessor] Adding technical indicators")
            
            # Initialize indicator strategy
            print("\n[DataProcessor] Setting up indicators:")
            for name, params in TECHNICAL_INDICATORS.items():
                print(f" - {name}: {params}")
            
            custom_strategy = ta.Strategy(
                name="custom_strategy",
                ta=[
                    {"kind": "rsi", "length": TECHNICAL_INDICATORS['RSI']['length']},
                    {"kind": "macd", "fast": TECHNICAL_INDICATORS['MACD']['fast'],
                     "slow": TECHNICAL_INDICATORS['MACD']['slow'],
                     "signal": TECHNICAL_INDICATORS['MACD']['signal']},
                    {"kind": "bbands", "length": TECHNICAL_INDICATORS['BB']['length'],
                     "std": TECHNICAL_INDICATORS['BB']['std']},
                    {"kind": "atr", "length": TECHNICAL_INDICATORS['ATR']['length']},
                    {"kind": "cci", "length": TECHNICAL_INDICATORS['CCI']['length']},
                    {"kind": "vhf", "length": TECHNICAL_INDICATORS['VHF']['length']},
                    {"kind": "eri", "length": TECHNICAL_INDICATORS['ERI']['length']},
                    {"kind": "adx", "length": TECHNICAL_INDICATORS['ADX']['length']}
                ]
            )
            
            # Store initial columns for comparison
            initial_columns = set(df.columns)
            
            # Calculate all indicators
            print("\n[DataProcessor] Calculating indicators...")
            df.ta.strategy(custom_strategy)
            
            # Print summary of added indicators
            new_columns = set(df.columns) - initial_columns
            print("\n[DataProcessor] Technical indicators added:")
            for col in sorted(new_columns):
                print(f" ✓ {col}")
            
            # Clean up and print final stats
            initial_rows = len(df)
            df = df.dropna()
            print(f"\n[DataProcessor] Final statistics:")
            print(f" - Initial rows: {initial_rows}")
            print(f" - Final rows: {len(df)}")
            print(f" - Added indicators: {len(new_columns)}")
            print(f" - Total features: {len(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error in technical analysis: {str(e)}")
            raise

: 

## Neural Network and Trading Agent
Define the DQN architecture and trading agent

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQNNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, output_size)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class TradingAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=100000)  # Fixed memory size
        self.batch_size = RL_CONFIG['batch_size']
        
        self.gamma = RL_CONFIG['gamma']
        self.epsilon = RL_CONFIG['epsilon_start']
        self.epsilon_min = RL_CONFIG['epsilon_end']
        self.epsilon_decay = RL_CONFIG['epsilon_decay']
        
        self.model = DQNNetwork(state_size, action_size)
        self.target_model = DQNNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=RL_CONFIG['learning_rate'])
        
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)
        
        state = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            action_values = self.model(state)
        return torch.argmax(action_values).item()

    def train(self):
        if len(self.memory) < self.batch_size:
            return
        
        batch = random.sample(self.memory, self.batch_size)
        states = torch.FloatTensor([i[0] for i in batch])
        actions = torch.LongTensor([i[1] for i in batch])
        rewards = torch.FloatTensor([i[2] for i in batch])
        next_states = torch.FloatTensor([i[3] for i in batch])
        dones = torch.FloatTensor([i[4] for i in batch])
        
        current_q_values = self.model(states).gather(1, actions.unsqueeze(1))
        next_q_values = self.target_model(next_states).max(1)[0].detach()
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        
        loss = nn.MSELoss()(current_q_values.squeeze(), target_q_values)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

: 

## Training Process
Set up training environment and train the model

In [ ]:
def setup_training(timeframe: str):
    data_file = DATA_DIR / f"{timeframe}.csv"
    processor = DataProcessor()
    df = processor.load_data(data_file)
    
    logger.info(f"Original dataframe shape: {df.shape}")
    logger.info(f"Original columns: {df.columns.tolist()}")
    
    df = processor.add_technical_indicators(df)
    df = df.dropna()
    
    logger.info(f"Processed dataframe shape: {df.shape}")
    logger.info(f"Processed columns: {df.columns.tolist()}")
    
    num_features = len(df.columns)
    env = ForexTradingEnv(df, state_dim=num_features)
    agent = TradingAgent(state_size=num_features, action_size=3)
    
    return env, agent, df

: 

In [ ]:
def train_model(timeframe: str, episodes: int = 1000):
    env, agent, df = setup_training(timeframe)
    best_reward = float('-inf')
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            action = agent.act(state)
            next_state, reward, done, _ = env.step(action)
            
            agent.remember(state, action, reward, next_state, done)
            agent.train()
            
            state = next_state
            total_reward += reward
        
        logger.info(f"Episode {episode + 1}/{episodes}, Total Reward: {total_reward:.2f}")
        
        if total_reward > best_reward:
            best_reward = total_reward
            model_path = MODELS_DIR / f"best_model_{timeframe}.pth"
            torch.save(agent.model.state_dict(), model_path)
            logger.info(f"New best model saved with reward: {best_reward:.2f}")

: 

In [ ]:
# Train models for different timeframes
timeframes = ['M5', 'M15', 'M30', 'H1', 'H4']

for timeframe in timeframes:
    logger.info(f"Starting training for {timeframe} timeframe")
    try:
        train_model(timeframe)
    except Exception as e:
        logger.error(f"Error training {timeframe}: {str(e)}")
        continue

INFO:__main__:Starting training for M5 timeframe



[DataProcessor] Initialized

[DataProcessor] Loading data from /home/chiqo/Documents/AI/High-Frequency-Trading-Model-with-IB/data/M5.csv

[DataProcessor] Reading data chunks:


Reading data: 100it [00:00, 223.92it/s]



[DataProcessor] Raw data shape: (100000, 7)

[DataProcessor] Processing steps:
1. Converting time...
2. Converting numeric columns...


INFO:__main__:Original dataframe shape: (288, 6)
INFO:__main__:Original columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']


...

INFO:__main__:Original dataframe shape: (6, 6)
INFO:__main__:Original columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']


3. Cleaning data...
4. Setting index...

[DataProcessor] Data loading completed:
 - Initial rows: 25850
 - Final rows: 6
 - Columns: Open, High, Low, Close, Volume, Spread
--------------------------------------------------

[DataProcessor] Adding technical indicators

[DataProcessor] Setting up indicators:
 - RSI: {'length': 7}
 - MACD: {'fast': 8, 'slow': 17, 'signal': 6}
 - BB: {'length': 10, 'std': 2}
 - ATR: {'length': 7}
 - CCI: {'length': 10}
 - VHF: {'length': 14}
 - ERI: {'length': 7}
 - ADX: {'length': 10}

[DataProcessor] Calculating indicators...


INFO:__main__:Processed dataframe shape: (6, 6)
INFO:__main__:Processed columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
INFO:__main__:Episode 1/1000, Total Reward: -0.00
INFO:__main__:New best model saved with reward: -0.00
INFO:__main__:Episode 2/1000, Total Reward: 0.00
INFO:__main__:New best model saved with reward: 0.00
INFO:__main__:Episode 3/1000, Total Reward: 0.00
INFO:__main__:Episode 4/1000, Total Reward: 0.00
INFO:__main__:Episode 5/1000, Total Reward: 0.01
INFO:__main__:New best model saved with reward: 0.01
INFO:__main__:Episode 6/1000, Total Reward: -0.01
ERROR:__main__:Error training H4: mat1 and mat2 shapes cannot be multiplied (32x8 and 6x64)



[DataProcessor] Technical indicators added:

[DataProcessor] Final statistics:
 - Initial rows: 6
 - Final rows: 6
 - Added indicators: 0
 - Total features: 6
--------------------------------------------------


: 